# Hands-On 7: Clustering and feature-space geometry

Record a prediction before running each experiment.

In [ ]:
from pathlib import Path
import os
import sys
try:
    import mlcourse.setup
except ModuleNotFoundError:
    bases = [Path(os.environ.get("MLCOURSE_ROOT", Path.cwd())), Path.cwd(), Path("/content/pp-machine-learning")]
    for base in bases:
        for candidate in (base.resolve(), *base.resolve().parents):
            if (candidate / "src/mlcourse/setup.py").is_file():
                sys.path.insert(0, str(candidate / "src"))
                break
        else:
            continue
        break
    else:
        raise RuntimeError("Course files not found. Open the extracted course repository or set MLCOURSE_ROOT to its location.") from None

In [ ]:
from mlcourse.setup import setup_notebook
REPO_ROOT = setup_notebook()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlcourse.labs import load_course_data

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram
from mlcourse.labs import compare_clusters, cluster_summary
from mlcourse.widgets import interactive_clustering
from mlcourse.labs import cluster_plot

## 1. Compare three partitions of a toy dataset

Load the eight points. Compare K-Means, DBSCAN and single-linkage clustering. K-Means uses Euclidean geometry while the other two use Manhattan distance here.

**Prediction:** Can density connectivity recover a different structure from centroid assignment?

*Your response.*

In [ ]:
toy = load_course_data('toy').set_index('id')[['A1', 'A2']]
toy_models = {'K-Means': KMeans(n_clusters=2, init=toy.loc[['p1', 'p6']].to_numpy(), n_init=1, random_state=0),
              'DBSCAN': DBSCAN(eps=1, min_samples=2, metric='manhattan'),
              'Single linkage': AgglomerativeClustering(n_clusters=2, linkage='single', metric='manhattan')}
display(toy)
display(compare_clusters(toy_models, toy))

**Observation:** Compare cluster membership and any noise points.

*Your response.*

**Explanation:** Explain the three rules without relying on the arbitrary cluster numbers.

*Your response.*

## 2. Scale a real feature space

Load Wholesale data and select `Grocery` and `Milk`. Compare K-Means before and after standardisation.

**Prediction:** How can feature scale influence Euclidean distances?

*Your response.*

In [ ]:
wholesale = load_course_data('wholesale')
raw = wholesale[['Grocery', 'Milk']]
scaled = pd.DataFrame(StandardScaler().fit_transform(raw), columns=raw.columns, index=raw.index)
display(raw.head())
fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout='constrained')
memberships = {}
for ax, values, name in zip(axes, [raw, scaled], ['Original units', 'Standardised']):
    labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(values)
    memberships[name] = labels
    cluster_plot(ax, values, labels, name)
plt.show()
display(pd.crosstab(pd.Series(memberships['Original units'], name='Original cluster'),
                    pd.Series(memberships['Standardised'], name='Scaled cluster')))

**Observation:** Locate points whose membership changes.

*Your response.*

**Explanation:** Why is scaling part of the clustering assumptions?

*Your response.*

## 3. Explore cluster number

Compare `n_clusters` from `2` to `8` using inertia and silhouette.

**Prediction:** Must inertia decrease when more centroids are available?

*Your response.*

In [ ]:
rows = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=10, random_state=0).fit(scaled)
    rows.append({'n_clusters': k, 'inertia': model.inertia_, **cluster_summary(scaled, model.labels_)})
cluster_scores = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
axes[0].plot(cluster_scores.n_clusters, cluster_scores.inertia, '-o')
axes[1].plot(cluster_scores.n_clusters, cluster_scores.silhouette, '-s')
axes[0].set(xlabel='n_clusters', ylabel='Inertia', title='Within-cluster squared distance')
axes[1].set(xlabel='n_clusters', ylabel='Silhouette', title='Separation and compactness')
plt.show()
display(cluster_scores[['n_clusters', 'inertia', 'silhouette']])

**Observation:** Compare the elbow suggestion with the largest silhouette.

*Your response.*

**Explanation:** Why can the two criteria suggest different choices?

*Your response.*

## 4. Inspect density clustering

Fit DBSCAN with `eps=0.500` and `min_samples=5` on the scaled data.

**Prediction:** What kinds of points are likely to be marked as noise?

*Your response.*

In [ ]:
display(compare_clusters({'DBSCAN': DBSCAN(eps=.5, min_samples=5)}, scaled))

**Observation:** Count the clusters and noise observations.

*Your response.*

**Explanation:** Explain how `eps` and `min_samples` define local support.

*Your response.*

## 5. Inspect hierarchical merging

Plot a truncated single-linkage dendrogram. Compare complete-linkage clustering in the laboratory below.

**Prediction:** What does a high merge in the dendrogram indicate?

*Your response.*

In [ ]:
merges = linkage(scaled, method='single', metric='euclidean')
fig, ax = plt.subplots(figsize=(11, 5), layout='constrained')
dendrogram(merges, truncate_mode='lastp', p=12, show_leaf_counts=True, ax=ax)
ax.set(xlabel='Point or collapsed group (count)', ylabel='Merge distance', title='Single-linkage merging, last 12 groups')
plt.show()

**Observation:** Identify a large change in merge distance.

*Your response.*

**Explanation:** How would complete linkage change the meaning of inter-cluster distance?

*Your response.*

## 6. Compare full-feature partitions

Use the six spending measurements, then display their partitions in two principal components. Inspect one compact profile table.

**Prediction:** Will a two-dimensional projection preserve every separation in six dimensions?

*Your response.*

In [ ]:
spending = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
full_scaled = StandardScaler().fit_transform(wholesale[spending])
pca = PCA(n_components=2).fit(full_scaled)
projection = pd.DataFrame(pca.transform(full_scaled), columns=['PC1', 'PC2'])
full_models = {'K-Means': KMeans(n_clusters=3, n_init=10, random_state=0),
               'DBSCAN': DBSCAN(eps=.8, min_samples=5),
               'Hierarchical': AgglomerativeClustering(n_clusters=3, linkage='complete')}
display(compare_clusters(full_models, full_scaled, projection))
print(f'Variance displayed: {pca.explained_variance_ratio_.sum():.3f}')
labels = full_models['K-Means'].fit_predict(full_scaled)
profiles = pd.DataFrame(full_scaled, columns=spending).groupby(labels).mean()
display(profiles)

**Observation:** Compare projected overlap and standardised profiles.

*Your response.*

**Explanation:** Why should the silhouette use the fitted feature space rather than the display coordinates?

*Your response.*

## 7. Manipulate clustering assumptions

Change `n_clusters`, DBSCAN neighbourhood parameters and hierarchical linkage on the same two-feature data.

**Prediction:** Which algorithm can leave observations unassigned?

*Your response.*

In [ ]:
clustering_lab = interactive_clustering(scaled)
display(clustering_lab.widget)

**Observation:** Record one change in cluster membership and one change in noise count.

*Your response.*

**Explanation:** Explain which geometric assumption caused each change.

*Your response.*